# LeNet-5 — Entrenamiento

In [ ]:
# CELDA 1 — Montar Drive y descomprimir
from google.colab import drive
drive.mount('/content/drive')

import subprocess
result = subprocess.run([
    'unzip', '-q',
    '/content/drive/MyDrive/Datasets/Dataset_P2_DL.zip',
    '-d', '/content/data'
])
print('Descompresión lista')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Descompresión lista


In [ ]:
# CELDA 2 — Limpiar archivos corruptos
from PIL import Image
import os

corrupted = []
for root, _, files in os.walk('/content/data'):
    for fname in files:
        fpath = os.path.join(root, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
        except Exception:
            corrupted.append(fpath)

print(f'Archivos corruptos: {len(corrupted)}')
for f in corrupted:
    print(f)
    os.remove(f)
print('Eliminados')

Archivos corruptos: 0
Eliminados


### Baseline

In [ ]:
# CELDA 3 — Arquitectura LeNet-5 adaptada
import torch
from torch import nn
from torch.nn import Conv2d, AvgPool2d, ReLU, Module, Linear, Dropout, BatchNorm2d

class Lenet5_224px(Module):
    def __init__(self, in_dim, n_classes):
        super(Lenet5_224px, self).__init__()
        # Capas convolucionales — igual que LeNet original
        self.conv1 = Conv2d(in_dim, 6,   kernel_size=5, padding=2)
        self.conv2 = Conv2d(6,      16,  kernel_size=5, padding=1)
        self.conv3 = Conv2d(16,     120, kernel_size=5, padding=0)

        self.pool = AvgPool2d(kernel_size=2, stride=2)
        self.act  = ReLU(inplace=True)

        # AdaptivePool para manejar el tamaño 224x224
        self.adapt_pool = nn.AdaptiveAvgPool2d((4, 4))

        # Capas FC
        self.fc1     = Linear(4 * 4 * 120, 84)
        self.dropout = Dropout(0.4)
        self.fc2     = Linear(84, n_classes)

    def forward(self, x):
        # Conv → Act → Pool  (orden correcto)
        x = self.act(self.conv1(x))
        x = self.pool(x)

        x = self.act(self.conv2(x))
        x = self.pool(x)

        x = self.act(self.conv3(x))
        x = self.adapt_pool(x)
        x = torch.flatten(x, 1)

        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print('Arquitectura definida')

Arquitectura definida


In [ ]:
# CELDA 4 — Imports, configuración, datos y loaders
import os, time, json, copy
from pathlib import Path
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# ── Hiperparámetros ─────────────────────────────────────────
SEED             = 42
EPOCHS           = 60
BATCH_SIZE       = 128
LR               = 0.01      # SGD
MOMENTUM         = 0.9
WEIGHT_DECAY     = 5e-4
NUM_WORKERS      = 4
CHECKPOINT_EVERY = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Rutas ───────────────────────────────────────────────────
DATA_PATH      = '/content/data/Data_subsampled'
OUT_DIR        = Path('/content/drive/MyDrive/Datasets/outputs_lenet_variante4')#cambiar aqui la salida si no se vaa sobreescribir
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
PLOTS_DIR      = OUT_DIR / 'plots'
METRICS_DIR    = OUT_DIR / 'metrics'
for p in (CHECKPOINT_DIR, PLOTS_DIR, METRICS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── Reproducibilidad ────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ── Media y std del dataset ──────────────────────────────────
def compute_mean_std(data_path):
    ds     = datasets.ImageFolder(data_path, transform=transforms.ToTensor())
    loader = DataLoader(ds, batch_size=128, num_workers=4, shuffle=False,
                        persistent_workers=True)
    mean = torch.zeros(3)
    std  = torch.zeros(3)
    for imgs, _ in loader:
        mean += imgs.mean(dim=[0, 2, 3])
        std  += imgs.std(dim=[0, 2, 3])
    mean /= len(loader)
    std  /= len(loader)
    print(f'Mean: {mean}  Std: {std}')
    return mean.tolist(), std.tolist()

MEAN, STD = compute_mean_std(DATA_PATH)

# ── Transforms ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Dataset y splits ────────────────────────────────────────
dataset      = datasets.ImageFolder(root=DATA_PATH, transform=train_transform)
dataset_eval = datasets.ImageFolder(root=DATA_PATH, transform=eval_transform)

class_names = dataset.classes
n_classes   = len(class_names)
print(f'Clases: {class_names}  |  Total: {len(dataset)}  |  Device: {DEVICE}')

targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, targets, train_size=0.8, stratify=targets, random_state=SEED)
val_idx, test_idx, _, _ = train_test_split(
    temp_idx, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

train_dataset = Subset(dataset,      train_idx)
val_dataset   = Subset(dataset_eval, val_idx)
test_dataset  = Subset(dataset_eval, test_idx)
print(f'train:{len(train_dataset)}  val:{len(val_dataset)}  test:{len(test_dataset)}')

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=True, persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kw)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kw)
test_loader  = DataLoader(test_dataset,  shuffle=False, **loader_kw)
print('Loaders listos')

Mean: tensor([0.3745, 0.3034, 0.4152])  Std: tensor([0.3549, 0.3974, 0.4199])
Clases: ['Disgust', 'Fear', 'Happy', 'Neutral', 'Sad']  |  Total: 30000  |  Device: cuda
train:24000  val:3000  test:3000
Loaders listos


In [ ]:
# CELDA 5 — Utilidades y funciones de entrenamiento

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(state, filename):
    torch.save(state, str(filename))

def save_curves(history, outdir):
    epochs = list(range(1, len(history['train_loss']) + 1))
    for keys, title, fname in [
        (('train_loss','val_loss'), 'Loss',     'loss_curve.png'),
        (('train_f1',  'val_f1'),  'Macro F1', 'f1_curve.png'),
    ]:
        plt.figure()
        for k in keys: plt.plot(epochs, history[k], label=k)
        plt.xlabel('Epoch'); plt.ylabel(title)
        plt.title(f'{title} - Train/Val'); plt.legend(); plt.grid(True)
        plt.savefig(outdir / fname, dpi=150); plt.close()
    import csv
    with open(outdir / 'metrics_per_epoch.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_loss','val_loss','train_f1','val_f1','epoch_time_s'])
        for i in range(len(epochs)):
            w.writerow([epochs[i], history['train_loss'][i], history['val_loss'][i],
                        history['train_f1'][i], history['val_f1'][i], history['epoch_times'][i]])

def save_confusion_matrix(y_true, y_pred, classes, outpath, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    cm_plot = cm.astype('float') / (cm.sum(axis=1)[:, None] + 1e-12) if normalize else cm
    plt.figure(figsize=(8, 6))
    plt.imshow(cm_plot, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de confusión'); plt.colorbar()
    ticks = np.arange(len(classes))
    plt.xticks(ticks, classes, rotation=45, ha='right')
    plt.yticks(ticks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm_plot.max() / 2.
    for i, j in np.ndindex(cm_plot.shape):
        plt.text(j, i, format(cm_plot[i, j], fmt), ha='center',
                 color='white' if cm_plot[i, j] > thresh else 'black')
    plt.ylabel('Clase real'); plt.xlabel('Clase predicha')
    plt.tight_layout(); plt.savefig(outpath, dpi=150); plt.close()
    np.save(str(outpath.with_suffix('.npy')), cm)

def run_epoch(model, loader, criterion, optimizer, device, training=True):
    model.train() if training else model.eval()
    losses, preds_all, tgts_all = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, tgts in loader:
            imgs, tgts = imgs.to(device), tgts.to(device)
            out  = model(imgs)
            loss = criterion(out, tgts)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            preds_all.extend(out.argmax(1).cpu().numpy())
            tgts_all.extend(tgts.cpu().numpy())
    avg_loss = float(np.mean(losses))
    f1  = float(f1_score(tgts_all, preds_all, average='macro', zero_division=0))
    acc = float(accuracy_score(tgts_all, preds_all))
    return avg_loss, f1, acc, np.array(preds_all), np.array(tgts_all)

def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5)

    history = {k: [] for k in ('train_loss','val_loss','train_f1','val_f1','epoch_times')}
    best_val_acc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        tr_loss, tr_f1, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, DEVICE, training=True)
        vl_loss, vl_f1, vl_acc, vp, vt = run_epoch(model, val_loader, criterion, None, DEVICE, training=False)
        scheduler.step(vl_loss)
        elapsed = time.perf_counter() - t0

        for k, v in zip(('train_loss','val_loss','train_f1','val_f1','epoch_times'),
                        (tr_loss, vl_loss, tr_f1, vl_f1, elapsed)):
            history[k].append(v)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{EPOCHS} | '
              f'tr_loss={tr_loss:.4f} tr_f1={tr_f1:.4f} tr_acc={tr_acc:.4f} | '
              f'vl_loss={vl_loss:.4f} vl_f1={vl_f1:.4f} vl_acc={vl_acc:.4f} | '
              f'lr={lr_now:.5f} | {elapsed:.1f}s')

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint({'epoch': epoch, 'model_state': model.state_dict(),
                             'optimizer_state': optimizer.state_dict(), 'history': history},
                            CHECKPOINT_DIR / f'checkpoint_epoch_{epoch:03d}.pth')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state = {'epoch': epoch, 'val_acc': vl_acc,
                          'model_state': copy.deepcopy(model.state_dict()),
                          'history': history}
            save_checkpoint(best_state, CHECKPOINT_DIR / 'best_model.pth')
            print(f'  ★ Mejor modelo (val_acc={vl_acc:.4f})')

    save_checkpoint({'epoch': EPOCHS, 'model_state': model.state_dict(), 'history': history},
                    CHECKPOINT_DIR / 'final_model.pth')
    return history, best_state

print('Funciones definidas')

Funciones definidas


In [ ]:
# CELDA 6 — Entrenamiento y evaluación final
model    = Lenet5_224px(in_dim=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history, best_state = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

criterion = nn.CrossEntropyLoss()

# Evaluación en validación
model.load_state_dict(best_state['model_state'])
model.to(DEVICE)
_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)
print(f'[VAL  best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')
save_confusion_matrix(vl_tgts, vl_preds, class_names, PLOTS_DIR / 'confusion_matrix_val.png')
report = classification_report(vl_tgts, vl_preds, target_names=class_names, digits=4, zero_division=0)
(METRICS_DIR / 'classification_report_val.txt').write_text(report)
np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# Evaluación en test
_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)
print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')
(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n')
np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

# Resumen final
summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state["val_acc"]:.4f}\n'
    f'best_val_f1={max(history["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)
(METRICS_DIR / 'final_summary.txt').write_text(summary)
print('\n' + summary)

Parámetros: 212,781 (0.213 M)
Ep   1/60 | tr_loss=1.6095 tr_f1=0.1874 tr_acc=0.2056 | vl_loss=1.6095 vl_f1=0.0730 vl_acc=0.2003 | lr=0.01000 | 23.6s
  ★ Mejor modelo (val_acc=0.2003)
Ep   2/60 | tr_loss=1.6085 tr_f1=0.1891 tr_acc=0.2051 | vl_loss=1.6093 vl_f1=0.1480 vl_acc=0.2097 | lr=0.01000 | 22.1s
  ★ Mejor modelo (val_acc=0.2097)
Ep   3/60 | tr_loss=1.6072 tr_f1=0.1924 tr_acc=0.2146 | vl_loss=1.6094 vl_f1=0.1652 vl_acc=0.1993 | lr=0.01000 | 22.5s
Ep   4/60 | tr_loss=1.6069 tr_f1=0.1853 tr_acc=0.2149 | vl_loss=1.6087 vl_f1=0.1619 vl_acc=0.2080 | lr=0.01000 | 22.3s
Ep   5/60 | tr_loss=1.6062 tr_f1=0.1985 tr_acc=0.2166 | vl_loss=1.6089 vl_f1=0.1419 vl_acc=0.2127 | lr=0.01000 | 20.6s
  ★ Mejor modelo (val_acc=0.2127)
Ep   6/60 | tr_loss=1.6058 tr_f1=0.1857 tr_acc=0.2193 | vl_loss=1.6108 vl_f1=0.1508 vl_acc=0.2067 | lr=0.01000 | 20.3s
Ep   7/60 | tr_loss=1.6054 tr_f1=0.1998 tr_acc=0.2182 | vl_loss=1.6090 vl_f1=0.1836 vl_acc=0.2147 | lr=0.01000 | 20.1s
  ★ Mejor modelo (val_acc=0.2147)
E

In [ ]:
# CELDA 7.-Variante1
import torch
from torch import nn
from torch.nn import Conv2d, AvgPool2d, ReLU, Module, Linear, Dropout, BatchNorm2d

class Lenet5_224px(Module):
    def __init__(self, in_dim, n_classes):
        super(Lenet5_224px, self).__init__()
        # Capas convolucionales — igual que LeNet original
        self.conv1 = Conv2d(in_dim, 6,   kernel_size=5, padding=2)
        self.conv2 = Conv2d(6,      16,  kernel_size=5, padding=1)
        self.conv3 = Conv2d(16,     120, kernel_size=5, padding=0)

        self.pool = AvgPool2d(kernel_size=2, stride=2)
        self.act  = ReLU(inplace=True)

        # AdaptivePool para manejar el tamaño 224x224
        self.adapt_pool = nn.AdaptiveAvgPool2d((4, 4))

        # Capas FC
        self.fc1     = Linear(4 * 4 * 120, 84)
        self.dropout = Dropout(0.4)
        self.fc2     = Linear(84, n_classes)

    def forward(self, x):
        # Conv → Act → Pool
        x = self.act(self.conv1(x))
        x = self.pool(x)

        x = self.act(self.conv2(x))
        x = self.pool(x)

        x = self.act(self.conv3(x))
        x = self.adapt_pool(x)

        x = torch.flatten(x, 1)

        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


print('Arquitectura baseline definida')


# ===============================
# VARIANTE L1 — LeNet + BatchNorm
# ===============================

class Lenet5_BN(Module):
    def __init__(self, in_dim, n_classes):
        super(Lenet5_BN, self).__init__()

        # Convoluciones con BatchNorm
        self.conv1 = Conv2d(in_dim, 6, kernel_size=5, padding=2)
        self.bn1   = BatchNorm2d(6)

        self.conv2 = Conv2d(6, 16, kernel_size=5, padding=1)
        self.bn2   = BatchNorm2d(16)

        self.conv3 = Conv2d(16, 120, kernel_size=5, padding=0)
        self.bn3   = BatchNorm2d(120)

        self.pool = AvgPool2d(kernel_size=2, stride=2)
        self.act  = ReLU(inplace=True)

        self.adapt_pool = nn.AdaptiveAvgPool2d((4,4))

        # FC
        self.fc1     = Linear(4 * 4 * 120, 84)
        self.dropout = Dropout(0.4)
        self.fc2     = Linear(84, n_classes)

    def forward(self, x):

        x = self.pool(self.act(self.bn1(self.conv1(x))))
        x = self.pool(self.act(self.bn2(self.conv2(x))))
        x = self.act(self.bn3(self.conv3(x)))

        x = self.adapt_pool(x)

        x = torch.flatten(x, 1)

        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


print('Variante L1 (LeNet + BatchNorm) definida')

Arquitectura baseline definida
Variante L1 (LeNet + BatchNorm) definida


In [ ]:
# CELDA 7 — Entrenamiento y evaluación final (Variante L1)

model    = Lenet5_BN(in_dim=3, n_classes=n_classes).to(DEVICE)
n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history_L1, best_state_L1 = train_model(model, train_loader, val_loader)

# Guardar curvas e historial
save_curves(history_L1, PLOTS_DIR)
with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history_L1, f, indent=2)

criterion = nn.CrossEntropyLoss()

# ===============================
# Evaluación en VALIDACIÓN
# ===============================

model.load_state_dict(best_state_L1['model_state'])
model.to(DEVICE)

_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)

print(f'[VAL best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')

save_confusion_matrix(
    vl_tgts, vl_preds, class_names,
    PLOTS_DIR / 'confusion_matrix_val.png'
)

report = classification_report(
    vl_tgts, vl_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
)

(METRICS_DIR / 'classification_report_val.txt').write_text(report)

np.save(METRICS_DIR / 'val_preds.npy',   vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)


# ===============================
# Evaluación en TEST
# ===============================

_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)

print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')

(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n'
)

np.save(METRICS_DIR / 'test_preds.npy',   ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)


# ===============================
# Resumen final
# ===============================

summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history_L1["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state_L1["val_acc"]:.4f}\n'
    f'best_val_f1={max(history_L1["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)

(METRICS_DIR / 'final_summary.txt').write_text(summary)

print('\n' + summary)

Parámetros: 213,065 (0.213 M)
Ep   1/60 | tr_loss=1.6109 tr_f1=0.1954 tr_acc=0.2047 | vl_loss=1.6090 vl_f1=0.1154 vl_acc=0.2010 | lr=0.01000 | 20.1s
  ★ Mejor modelo (val_acc=0.2010)
Ep   2/60 | tr_loss=1.6077 tr_f1=0.1696 tr_acc=0.2112 | vl_loss=1.6083 vl_f1=0.1301 vl_acc=0.2120 | lr=0.01000 | 20.2s
  ★ Mejor modelo (val_acc=0.2120)
Ep   3/60 | tr_loss=1.6066 tr_f1=0.1630 tr_acc=0.2205 | vl_loss=1.6079 vl_f1=0.1577 vl_acc=0.2100 | lr=0.01000 | 20.1s
Ep   4/60 | tr_loss=1.6062 tr_f1=0.1764 tr_acc=0.2166 | vl_loss=1.6080 vl_f1=0.1499 vl_acc=0.2063 | lr=0.01000 | 20.0s
Ep   5/60 | tr_loss=1.6064 tr_f1=0.1882 tr_acc=0.2155 | vl_loss=1.6087 vl_f1=0.1740 vl_acc=0.2057 | lr=0.01000 | 20.0s
Ep   6/60 | tr_loss=1.6065 tr_f1=0.1888 tr_acc=0.2155 | vl_loss=1.6079 vl_f1=0.1262 vl_acc=0.2100 | lr=0.01000 | 20.0s
Ep   7/60 | tr_loss=1.6066 tr_f1=0.2013 tr_acc=0.2158 | vl_loss=1.6086 vl_f1=0.1633 vl_acc=0.2080 | lr=0.01000 | 20.0s
Ep   8/60 | tr_loss=1.6059 tr_f1=0.1960 tr_acc=0.2174 | vl_loss=1.607

In [ ]:
#Variante L2: Adam

def train_model_L2(model, train_loader, val_loader):

    criterion = nn.CrossEntropyLoss()

    # CAMBIO DE LA VARIANTE L2
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
        "train_f1": [],
        "val_f1": [],
        "epoch_times": []
    }

    best_state = {
        "val_acc": 0,
        "model_state": None
    }

    for epoch in range(EPOCHS):

        start = time.time()

        # ================= TRAIN =================
        model.train()

        train_loss = 0
        train_preds = []
        train_targets = []

        for x, y in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(x)

            loss = criterion(outputs, y)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            train_preds.extend(preds.cpu().numpy())
            train_targets.extend(y.cpu().numpy())

        train_loss /= len(train_loader)

        train_acc = np.mean(np.array(train_preds) == np.array(train_targets))
        train_f1  = f1_score(train_targets, train_preds, average='macro')

        # ================= VALIDACIÓN =================
        model.eval()

        val_loss = 0
        val_preds = []
        val_targets = []

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(DEVICE)
                y = y.to(DEVICE)

                outputs = model(x)

                loss = criterion(outputs, y)

                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)

                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(y.cpu().numpy())

        val_loss /= len(val_loader)

        val_acc = np.mean(np.array(val_preds) == np.array(val_targets))
        val_f1  = f1_score(val_targets, val_preds, average='macro')

        epoch_time = time.time() - start

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1)

        history["epoch_times"].append(epoch_time)

        print(
            f'Epoch {epoch+1}/{EPOCHS} '
            f'Train Loss {train_loss:.4f} '
            f'Val Loss {val_loss:.4f} '
            f'Val Acc {val_acc:.4f}'
        )

        if val_acc > best_state["val_acc"]:
            best_state["val_acc"] = val_acc
            best_state["model_state"] = model.state_dict()

    return history, best_state

In [ ]:
# Entrenamiento y evaluación final (Variante L2)

model    = Lenet5_224px(in_dim=3, n_classes=n_classes).to(DEVICE)

n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history_L2, best_state_L2 = train_model_L2(model, train_loader, val_loader)

# Guardar curvas
save_curves(history_L2, PLOTS_DIR)

with open(METRICS_DIR / 'history.json', 'w') as f:
    json.dump(history_L2, f, indent=2)

criterion = nn.CrossEntropyLoss()

# ================= VALIDACIÓN =================

model.load_state_dict(best_state_L2['model_state'])

_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)

print(f'[VAL best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')

save_confusion_matrix(
    vl_tgts,
    vl_preds,
    class_names,
    PLOTS_DIR / 'confusion_matrix_val.png'
)

report = classification_report(
    vl_tgts,
    vl_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
)

(METRICS_DIR / 'classification_report_val.txt').write_text(report)

np.save(METRICS_DIR / 'val_preds.npy', vl_preds)
np.save(METRICS_DIR / 'val_targets.npy', vl_tgts)

# ================= TEST =================

_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)

print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')

(METRICS_DIR / 'test_metrics.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n'
)

np.save(METRICS_DIR / 'test_preds.npy', ts_preds)
np.save(METRICS_DIR / 'test_targets.npy', ts_tgts)

summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history_L2["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state_L2["val_acc"]:.4f}\n'
    f'best_val_f1={max(history_L2["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)

(METRICS_DIR / 'final_summary.txt').write_text(summary)

print('\n' + summary)

Parámetros: 212,781 (0.213 M)
Epoch 1/60 Train Loss 1.6091 Val Loss 1.6094 Val Acc 0.2083
Epoch 2/60 Train Loss 1.6078 Val Loss 1.6101 Val Acc 0.2020
Epoch 3/60 Train Loss 1.6069 Val Loss 1.6129 Val Acc 0.2050
Epoch 4/60 Train Loss 1.6068 Val Loss 1.6084 Val Acc 0.2043
Epoch 5/60 Train Loss 1.6065 Val Loss 1.6083 Val Acc 0.2093
Epoch 6/60 Train Loss 1.6064 Val Loss 1.6091 Val Acc 0.2087
Epoch 7/60 Train Loss 1.6061 Val Loss 1.6078 Val Acc 0.2073
Epoch 8/60 Train Loss 1.6055 Val Loss 1.6085 Val Acc 0.2023
Epoch 9/60 Train Loss 1.6062 Val Loss 1.6078 Val Acc 0.2147
Epoch 10/60 Train Loss 1.6058 Val Loss 1.6077 Val Acc 0.2063
Epoch 11/60 Train Loss 1.6056 Val Loss 1.6077 Val Acc 0.2093
Epoch 12/60 Train Loss 1.6056 Val Loss 1.6071 Val Acc 0.2077
Epoch 13/60 Train Loss 1.6050 Val Loss 1.6069 Val Acc 0.2080
Epoch 14/60 Train Loss 1.6058 Val Loss 1.6071 Val Acc 0.2077
Epoch 15/60 Train Loss 1.6048 Val Loss 1.6070 Val Acc 0.2103
Epoch 16/60 Train Loss 1.6051 Val Loss 1.6067 Val Acc 0.2083
Epo

In [ ]:
# ===============================
# Variante L3 – LeNet + canales amplios
# ===============================

class Lenet5_L3_224px(nn.Module):

    def __init__(self, in_dim=3, n_classes=10):
        super().__init__()

        self.features = nn.Sequential(

            # Conv1: 6 → 32
            nn.Conv2d(in_dim, 32, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(2),

            # Conv2: 16 → 64
            nn.Conv2d(32, 64, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(2),

            # reducir tamaño para 224px
            nn.AdaptiveAvgPool2d((4,4))
        )

        self.classifier = nn.Sequential(

            # FC1: 120 → 256
            nn.Linear(64*4*4, 256),
            nn.ReLU(),

            nn.Linear(256, 84),
            nn.ReLU(),

            nn.Linear(84, n_classes)
        )


    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x

In [ ]:
# Entrenamiento y evaluación final (Variante L3)

model = Lenet5_L3_224px(in_dim=3, n_classes=n_classes).to(DEVICE)

n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

history_L3, best_state_L3 = train_model(model, train_loader, val_loader)

# Guardar curvas
save_curves(history_L3, PLOTS_DIR)

with open(METRICS_DIR / 'history_L3.json', 'w') as f:
    json.dump(history_L3, f, indent=2)

criterion = nn.CrossEntropyLoss()

# ================= VALIDACIÓN =================

model.load_state_dict(best_state_L3['model_state'])

_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)

print(f'[VAL best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')

save_confusion_matrix(
    vl_tgts,
    vl_preds,
    class_names,
    PLOTS_DIR / 'confusion_matrix_val_L3.png'
)

report = classification_report(
    vl_tgts,
    vl_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
)

(METRICS_DIR / 'classification_report_val_L3.txt').write_text(report)

np.save(METRICS_DIR / 'val_preds_L3.npy', vl_preds)
np.save(METRICS_DIR / 'val_targets_L3.npy', vl_tgts)

# ================= TEST =================

_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)

print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')

(METRICS_DIR / 'test_metrics_L3.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n'
)

np.save(METRICS_DIR / 'test_preds_L3.npy', ts_preds)
np.save(METRICS_DIR / 'test_targets_L3.npy', ts_tgts)

summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history_L3["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state_L3["val_acc"]:.4f}\n'
    f'best_val_f1={max(history_L3["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)

(METRICS_DIR / 'final_summary_L3.txt').write_text(summary)

print('\n' + summary)

Parámetros: 338,109 (0.338 M)
Ep   1/60 | tr_loss=1.6093 tr_f1=0.1885 tr_acc=0.2079 | vl_loss=1.6092 vl_f1=0.1191 vl_acc=0.2067 | lr=0.01000 | 20.7s
  ★ Mejor modelo (val_acc=0.2067)
Ep   2/60 | tr_loss=1.6081 tr_f1=0.1911 tr_acc=0.2092 | vl_loss=1.6088 vl_f1=0.1582 vl_acc=0.1940 | lr=0.01000 | 20.0s
Ep   3/60 | tr_loss=1.6066 tr_f1=0.1982 tr_acc=0.2153 | vl_loss=1.6088 vl_f1=0.1573 vl_acc=0.2030 | lr=0.01000 | 19.9s
Ep   4/60 | tr_loss=1.6061 tr_f1=0.1906 tr_acc=0.2170 | vl_loss=1.6093 vl_f1=0.1545 vl_acc=0.2097 | lr=0.01000 | 20.0s
  ★ Mejor modelo (val_acc=0.2097)
Ep   5/60 | tr_loss=1.6052 tr_f1=0.2085 tr_acc=0.2226 | vl_loss=1.6084 vl_f1=0.1660 vl_acc=0.2090 | lr=0.01000 | 19.8s
Ep   6/60 | tr_loss=1.6049 tr_f1=0.2088 tr_acc=0.2219 | vl_loss=1.6091 vl_f1=0.1614 vl_acc=0.2027 | lr=0.01000 | 19.9s
Ep   7/60 | tr_loss=1.6048 tr_f1=0.2013 tr_acc=0.2199 | vl_loss=1.6094 vl_f1=0.1888 vl_acc=0.2110 | lr=0.01000 | 19.8s
  ★ Mejor modelo (val_acc=0.2110)
Ep   8/60 | tr_loss=1.6047 tr_f1=0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ===============================
# Variante L4 – LeNet + BN + canales amplios
# ===============================

class Lenet5_L4_224px(nn.Module):

    def __init__(self, in_dim=3, n_classes=10):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(in_dim, 32, kernel_size=5),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(32, 64, kernel_size=5),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.AdaptiveAvgPool2d((4,4))
        )

        self.classifier = nn.Sequential(

            nn.Linear(64*4*4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 84),
            nn.ReLU(),

            nn.Linear(84, n_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x

In [ ]:
# Entrenamiento y evaluación final (Variante L4)

model = Lenet5_L4_224px(in_dim=3, n_classes=n_classes).to(DEVICE)

n_params = count_parameters(model)
print(f'Parámetros: {n_params:,} ({n_params/1e6:.3f} M)')

# Optimizador AdamW para L4
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

history_L4, best_state_L4 = train_model(model, train_loader, val_loader)

# Guardar curvas
save_curves(history_L4, PLOTS_DIR)

with open(METRICS_DIR / 'history_L4.json', 'w') as f:
    json.dump(history_L4, f, indent=2)

criterion = nn.CrossEntropyLoss()

# ================= VALIDACIÓN =================

model.load_state_dict(best_state_L4['model_state'])

_, vl_f1, vl_acc, vl_preds, vl_tgts = run_epoch(
    model, val_loader, criterion, None, DEVICE, training=False)

print(f'[VAL best] acc={vl_acc:.4f}  f1={vl_f1:.4f}')

save_confusion_matrix(
    vl_tgts,
    vl_preds,
    class_names,
    PLOTS_DIR / 'confusion_matrix_val_L4.png'
)

report = classification_report(
    vl_tgts,
    vl_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
)

(METRICS_DIR / 'classification_report_val_L4.txt').write_text(report)

np.save(METRICS_DIR / 'val_preds_L4.npy', vl_preds)
np.save(METRICS_DIR / 'val_targets_L4.npy', vl_tgts)

# ================= TEST =================

_, ts_f1, ts_acc, ts_preds, ts_tgts = run_epoch(
    model, test_loader, criterion, None, DEVICE, training=False)

print(f'[TEST best] acc={ts_acc:.4f}  f1={ts_f1:.4f}')

(METRICS_DIR / 'test_metrics_L4.txt').write_text(
    f'test_acc={ts_acc:.4f}\ntest_f1={ts_f1:.4f}\n'
)

np.save(METRICS_DIR / 'test_preds_L4.npy', ts_preds)
np.save(METRICS_DIR / 'test_targets_L4.npy', ts_tgts)

summary = (
    f'n_parameters={n_params}\n'
    f'avg_time_per_epoch={np.mean(history_L4["epoch_times"]):.2f}s\n'
    f'best_val_acc={best_state_L4["val_acc"]:.4f}\n'
    f'best_val_f1={max(history_L4["val_f1"]):.4f}\n'
    f'test_acc={ts_acc:.4f}\n'
    f'test_f1={ts_f1:.4f}\n'
)

(METRICS_DIR / 'final_summary_L4.txt').write_text(summary)

print('\n' + summary)

Parámetros: 338,813 (0.339 M)
Ep   1/60 | tr_loss=1.6098 tr_f1=0.2141 tr_acc=0.2162 | vl_loss=1.6128 vl_f1=0.2074 vl_acc=0.2123 | lr=0.01000 | 22.3s
  ★ Mejor modelo (val_acc=0.2123)
Ep   2/60 | tr_loss=1.6052 tr_f1=0.2174 tr_acc=0.2198 | vl_loss=1.6099 vl_f1=0.1910 vl_acc=0.2083 | lr=0.01000 | 21.9s
Ep   3/60 | tr_loss=1.6035 tr_f1=0.2186 tr_acc=0.2230 | vl_loss=1.6094 vl_f1=0.1981 vl_acc=0.2163 | lr=0.01000 | 22.1s
  ★ Mejor modelo (val_acc=0.2163)
Ep   4/60 | tr_loss=1.6025 tr_f1=0.2239 tr_acc=0.2285 | vl_loss=1.6105 vl_f1=0.2076 vl_acc=0.2160 | lr=0.01000 | 22.0s
Ep   5/60 | tr_loss=1.6014 tr_f1=0.2221 tr_acc=0.2306 | vl_loss=1.6083 vl_f1=0.1910 vl_acc=0.2073 | lr=0.01000 | 22.2s
Ep   6/60 | tr_loss=1.6006 tr_f1=0.2292 tr_acc=0.2339 | vl_loss=1.6099 vl_f1=0.2007 vl_acc=0.2100 | lr=0.01000 | 21.9s
Ep   7/60 | tr_loss=1.6003 tr_f1=0.2273 tr_acc=0.2325 | vl_loss=1.6121 vl_f1=0.2015 vl_acc=0.2133 | lr=0.01000 | 22.1s
Ep   8/60 | tr_loss=1.5999 tr_f1=0.2265 tr_acc=0.2337 | vl_loss=1.608